# Gold: patient_360

Builds `<catalog>.gold.patient_360` (one row per patient) from silver, following STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.patients`, LEFT JOINed to hospitals, insurance_providers and per-patient lab_results aggregates. Full rebuild (CREATE OR REPLACE TABLE AS SELECT).

Note: `silver.hospitals` currently has duplicate rows per `hospital_id`, so it is deduplicated (latest `updated_timestamp`) before joining to preserve the grain.

In [0]:
import re

dbutils.widgets.text("catalog", "adb_carsync", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

# Guard against injection: catalog must be a plain identifier
if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: {catalog}")

target_table = f"{catalog}.gold.patient_360"
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH hospitals_dedup AS (
  -- silver.hospitals has duplicate rows per hospital_id; keep latest to preserve grain
  SELECT hospital_id, hospital_name
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY updated_timestamp DESC) AS rn
    FROM {catalog}.silver.hospitals
  )
  WHERE rn = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM {catalog}.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                     AS date_of_birth,
  CAST(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                     AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name                             AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)          AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)  AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)  AS critical_test_count,
  l.last_test_date
FROM {catalog}.silver.patients p
LEFT JOIN hospitals_dedup h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {catalog}.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id
""")

In [0]:
updated_df = spark.table(target_table)

record_count = updated_df.count()
print("Processed Record Count:", record_count)

dbutils.notebook.exit(str(record_count))